In [3]:
!apt-get install graphviz
!pip install graphviz

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-9ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 0 not upgraded.


In [ ]:
import os
import time
import graphviz
from google.colab import drive

drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/Trabalho_Arvores/Kd_tree'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Pasta de destino pronta em: {OUTPUT_DIR}")

In [ ]:
class KDNode:
    def __init__(self, point, depth=0):
        self.point = point
        self.left = None
        self.right = None

class KDTreeInstrumentada:
    def __init__(self, k=2):
        self.k = k
        self.root = None
        self.comparisons_count = 0
        self.nodes_count = 0

    def reset_metrics(self):
        self.comparisons_count = 0

    def insert(self, point):
        def _insert(node, point, depth):
            if node is None:
                self.nodes_count += 1
                return KDNode(point, depth)

            cd = depth % self.k
            self.comparisons_count += 1

            if point[cd] < node.point[cd]:
                node.left = _insert(node.left, point, depth + 1)
            else:
                node.right = _insert(node.right, point, depth + 1)

            return node

        self.root = _insert(self.root, point, 0)

    def search(self, point) -> bool:
        def _search(node, point, depth):
            if node is None:
                return False
            self.comparisons_count += 1
            if node.point == point:
                return True

            cd = depth % self.k
            if point[cd] < node.point[cd]:
                return _search(node.left, point, depth + 1)
            else:
                return _search(node.right, point, depth + 1)

        return _search(self.root, point, 0)

    def delete(self, point):
        def _find_min(node, d, depth):
            if node is None:
                return None

            cd = depth % self.k
            if cd == d:
                if node.left is None:
                    return node
                return _find_min(node.left, d, depth + 1)

            left_min = _find_min(node.left, d, depth + 1)
            right_min = _find_min(node.right, d, depth + 1)

            res = node
            if left_min and left_min.point[d] < res.point[d]:
                res = left_min
            if right_min and right_min.point[d] < res.point[d]:
                res = right_min
            return res

        def _delete(node, point, depth):
            if node is None:
                return None

            cd = depth % self.k
            self.comparisons_count += 1

            if node.point == point:
                self.nodes_count -= 1
                if node.right is not None:
                    min_node = _find_min(node.right, cd, depth + 1)
                    node.point = min_node.point
                    node.right = _delete(node.right, min_node.point, depth + 1)
                elif node.left is not None:
                    min_node = _find_min(node.left, cd, depth + 1)
                    node.point = min_node.point
                    node.right = _delete(node.right, min_node.point, depth + 1)
                    node.left = None
                else:
                    return None
                return node

            if point[cd] < node.point[cd]:
                node.left = _delete(node.left, point, depth + 1)
            else:
                node.right = _delete(node.right, point, depth + 1)

            return node

        self.root = _delete(self.root, point, 0)

    def save_diagram(self, filename: str, directory: str = OUTPUT_DIR):
        dot = graphviz.Digraph(comment='KD-Tree')
        axes = ['X', 'Y', 'Z', 'W']

        def _add_edges(node, depth=0):
            if node:
                node_id = str(id(node))
                axis_name = axes[depth % self.k]
                label = f"P: {node.point}\nEixo: {axis_name}"
                dot.node(node_id, label=label, shape="ellipse")

                if node.left:
                    left_id = str(id(node.left))
                    dot.edge(node_id, left_id, label="<")
                    _add_edges(node.left, depth + 1)
                if node.right:
                    right_id = str(id(node.right))
                    dot.edge(node_id, right_id, label=">=")
                    _add_edges(node.right, depth + 1)

        if self.root:
            _add_edges(self.root, 0)

        filepath = dot.render(filename=filename, directory=directory, format='png', cleanup=True)
        print(f"Diagrama KD-Tree salvo em: {filepath}")
        return dot

In [ ]:
kd = KDTreeInstrumentada(k=2)
pontos_exemplo = [(30, 40), (50, 30), (70, 60), (10, 20), (80, 10)]
for pt in pontos_exemplo:
    kd.insert(pt)

dot_kd = kd.save_diagram("kdtree_teste", directory=OUTPUT_DIR)
display(dot_kd)